# Any CRS

Pass the CRS of the axes with `crs` (anything `pyproj.CRS.from_user_input` reads, such as a GeoDataFrame's `crs`). The style is drawn in that CRS directly, so labels stay upright and sharp. Here, the capitals of Europe in the European LAEA projection (EPSG:3035) on CARTO's Positron.

In [ ]:
%pip install -q basemaplibre geopandas contextily

In [ ]:
import geopandas
import matplotlib.pyplot as plt

import basemaplibre as bml

In [ ]:
NE = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/"
places = geopandas.read_file(NE + "ne_50m_populated_places_simple.geojson")
capitals = places[places.featurecla == "Admin-0 capital"].cx[-12:32, 35:61]

capitals_laea = capitals.to_crs(3035)
ax = capitals_laea.plot(figsize=(6, 5.25), color="crimson", markersize=capitals.pop_max / 5e4)
bml.add_basemap(ax, crs=capitals_laea.crs, source="CartoDB.Positron")

## Longitude and latitude

EPSG:4326, as GeoPandas plots it: here, the largest cities of Japan. GeoPandas stretches the latitudes (by 1 / cos of the middle latitude) to keep the shape of the map; the style is drawn in that stretched projection, so labels keep theirs.

In [ ]:
japan = places[places.adm0name == "Japan"].cx[128:146, 30:46]
ax = japan.plot(figsize=(5, 6), color="crimson", markersize=japan.pop_max / 1e5)
bml.add_basemap(ax, crs=japan.crs, source="OpenFreeMap.Bright")

## The world

The world's capitals in Equal Earth (EPSG:8857). Beyond the edge of the projection the image is transparent.

In [ ]:
countries = geopandas.read_file(NE + "ne_110m_admin_0_countries.geojson")
world = places[places.featurecla == "Admin-0 capital"].to_crs(8857)

ax = world.plot(figsize=(8, 4.4), color="crimson", markersize=world.pop_max / 2e5)
ax.axis(countries.to_crs(8857).total_bounds[[0, 2, 1, 3]])
ax.set_axis_off()
bml.add_basemap(ax, crs=world.crs, source="Maptoolkit.Summer")

## Compared with contextily

contextily downloads raster tiles in Web Mercator and warps them to the CRS of the axes, which turns and blurs their labels; `basemaplibre` draws the style in that CRS. The defaults of each, on the capitals in EPSG:3035:

In [ ]:
import contextily as cx

fig, axs = plt.subplots(1, 2, figsize=(9, 4.4), layout="constrained")
for ax, title in zip(axs, ["contextily", "basemaplibre"], strict=True):
    capitals_laea.plot(ax=ax, color="crimson", markersize=capitals.pop_max / 5e4)
    ax.set_axis_off()
    ax.set_title(title)
cx.add_basemap(axs[0], crs=capitals_laea.crs)
bml.add_basemap(axs[1], crs=capitals_laea.crs)